# 1. Overview

Operator transaction log for one single unseen test sample diagnostic. This notebook resolves model identity from model run metadata, selects the requested checkpoint without fallback, runs one test sample, renders source video plus generated pose, writes reports, and publishes outputs. This is not full evaluation and not aggregate model performance.

# 2. Operator Configuration

## 2.1 Repository and roots

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/0xmillennium/text-to-sign-production.git"
REPO_REF = "chore/core-layout-notebook-workflows"
PROJECT_ROOT = Path("/content/text-to-sign-production")
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/text-to-sign-production")

## 2.2 Model test inputs

In [ ]:
MODEL_RUN_NAME = "paste-generated-model-run-here"
CHECKPOINT_POLICY = "best"
TARGET_SENTENCE_NAME = "paste-test-source-sentence-name-here"

## 2.3 Configuration review

In [ ]:
print(f"Repository URL: {REPO_URL}")
print(f"Repository ref: {REPO_REF}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Drive project root: {DRIVE_PROJECT_ROOT}")
print(f"MODEL_RUN_NAME: {MODEL_RUN_NAME}")
print(f"CHECKPOINT_POLICY: {CHECKPOINT_POLICY}")
print(f"TARGET_SENTENCE_NAME: {TARGET_SENTENCE_NAME}")
print("Scope: single unseen test sample diagnostic, not full evaluation or aggregate performance.")

# 3. Bootstrap Boundary

## 3.1 Mount Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
if not DRIVE_PROJECT_ROOT.parent.is_dir():
    raise FileNotFoundError(f"Drive project parent is missing: {DRIVE_PROJECT_ROOT.parent}")
print(f"Drive mounted: {DRIVE_PROJECT_ROOT.parent}")


## 3.2 System packages

In [ ]:
import shutil

if shutil.which("zstd") is None:
    !sudo apt-get update
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("Failed to update apt package index.")

    !sudo apt-get install -y zstd
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("Failed to install zstd.")
    print("Installed zstd.")
else:
    print("zstd is already available.")


## 3.3 Repository checkout

In [ ]:
%cd /content

if PROJECT_ROOT.exists():
    !rm -rf "{PROJECT_ROOT}"
    _exit_code = _exit_code if "_exit_code" in globals() else 0
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("stale checkout removal failed")

!git clone "{REPO_URL}" "{PROJECT_ROOT}"
_exit_code = _exit_code if "_exit_code" in globals() else 0
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("git clone failed")

%cd {PROJECT_ROOT}
!git checkout "{REPO_REF}"
_exit_code = _exit_code if "_exit_code" in globals() else 0
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("git checkout failed")

!git rev-parse HEAD
_exit_code = _exit_code if "_exit_code" in globals() else 0
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("rev-parse HEAD failed")

## 3.4 Install dependencies

In [ ]:
%cd {PROJECT_ROOT}
%pip install --upgrade pip
%pip install -r "requirements-colab.txt"

## 3.5 Add source tree to import path

In [ ]:
import sys

source_path = PROJECT_ROOT / "src"
if str(source_path) not in sys.path:
    sys.path.insert(0, str(source_path))
print(f"Source path active: {source_path}")

## 3.6 Workflow API import

In [ ]:
from text_to_sign_production.workflows.foundation.review import display_review_sections
from text_to_sign_production.workflows.test_model import (
    CheckpointPolicy,
    TestModelRequest,
    TestModelWorkflow,
    TestModelWorkflowConfig,
)
from text_to_sign_production.workflows.test_model.progress import (
    visible_test_model_progress_session,
)

# 4. Runtime Console: Plan

## 4.1 Build workflow config

In [ ]:
test_config = TestModelWorkflowConfig(
    project_root=PROJECT_ROOT,
    drive_project_root=DRIVE_PROJECT_ROOT,
)

## 4.2 Build test request

In [ ]:
test_request = TestModelRequest(
    model_run_name=MODEL_RUN_NAME,
    checkpoint_policy=CheckpointPolicy(CHECKPOINT_POLICY),
    target_sentence_name=TARGET_SENTENCE_NAME,
)

## 4.3 Instantiate workflow

In [ ]:
test_workflow = TestModelWorkflow(test_config)
test_model_progress = visible_test_model_progress_session(None)
display_review_sections(test_workflow.review_request(test_request))

## 4.4 Preflight readiness

Preflight checks the selected model run, checkpoint policy, target sentence, Drive inputs, provider support artifacts, and expected output roots before restore or inference starts.

In [ ]:
test_model_preflight = test_workflow.run_preflight(test_request)
display_review_sections(test_workflow.review_preflight(test_model_preflight))
if not test_model_preflight.ready:
    first_issue = test_model_preflight.blocking_issues[0] if test_model_preflight.blocking_issues else "unknown"
    raise RuntimeError(
        f"Test-model preflight failed with {len(test_model_preflight.blocking_issues)} blocking issue(s); "
        f"first issue: {first_issue}"
    )

## 4.5 Smoke execution protocol

Review the ordered smoke execution protocol before any restore, inference, or publish step is allowed to run.

In [ ]:
test_model_smoke_protocol = test_workflow.build_smoke_execution_protocol(
    test_request,
    test_model_preflight,
)
display_review_sections(test_workflow.review_smoke_execution_protocol(test_model_smoke_protocol))

## 4.4 Build runtime restore plan

In [ ]:
restore_plan = test_workflow.build_restore_plan(test_request)
print("test_model restore plan built.")

## 4.5 Review runtime restore plan

In [ ]:
display_review_sections(test_workflow.review_restore_plan(restore_plan))

## 4.6 Validate runtime restore plan

In [ ]:
restore_plan_validation = test_workflow.validate_restore_plan(restore_plan)
display_review_sections(test_workflow.review_restore_plan_validation(restore_plan_validation))
if not restore_plan_validation.valid:
    raise RuntimeError("Restore plan validation failed; stopping before restore.")

# 5. Runtime Console: Restore and Verify

## 5.1 Execute runtime restore

In [ ]:
restore_result = test_workflow.restore_runtime(
    restore_plan,
    progress_session=test_model_progress,
)
print("test_model runtime restore complete.")

## 5.2 Review restore result

In [ ]:
display_review_sections(test_workflow.review_restore_result(restore_result))

## 5.3 Verify runtime

In [ ]:
runtime_verification = test_workflow.verify_runtime(restore_plan)
print(f"Runtime ready: {runtime_verification.succeeded}")

## 5.4 Review runtime verification

In [ ]:
display_review_sections(test_workflow.review_runtime_verification(runtime_verification))
if not restore_result.succeeded or not runtime_verification.succeeded:
    raise RuntimeError("Runtime restore or verification failed; stopping before sample test.")

# 6. Runtime Console: Model Run and Checkpoint

## 6.1 Resolve model run

In [ ]:
model_run_resolution = test_workflow.resolve_model_run(restore_plan)

## 6.2 Review model run resolution

In [ ]:
display_review_sections(test_workflow.review_model_run_resolution(model_run_resolution))
if not model_run_resolution.succeeded:
    raise RuntimeError("Model run resolution failed; stopping before checkpoint selection.")

## 6.3 Select checkpoint

In [ ]:
checkpoint_selection = test_workflow.select_checkpoint(
    model_run_resolution,
    test_request.checkpoint_policy,
)

## 6.4 Review checkpoint selection

In [ ]:
display_review_sections(test_workflow.review_checkpoint_selection(checkpoint_selection))
if not checkpoint_selection.succeeded:
    raise RuntimeError("Checkpoint selection failed; stopping before target resolution.")

# 7. Runtime Console: Target Sample Evidence

## 7.1 Resolve target sample

In [ ]:
target_resolution = test_workflow.resolve_target_sample(
    test_request,
    model_run_resolution,
    progress_session=test_model_progress,
)

## 7.2 Review target resolution

In [ ]:
display_review_sections(test_workflow.review_target_resolution(target_resolution))
if not target_resolution.succeeded:
    raise RuntimeError("Target sample resolution failed; stopping before evidence collection.")

## 7.3 Collect sample evidence

In [ ]:
sample_evidence = test_workflow.collect_sample_evidence(
    model_run=model_run_resolution,
    checkpoint=checkpoint_selection,
    target=target_resolution,
    progress_session=test_model_progress,
)

## 7.4 Review sample evidence dossier

In [ ]:
display_review_sections(test_workflow.review_sample_evidence(sample_evidence))

# 8. Runtime Console: Single-Sample Model Test

## 8.1 Run model inference for target sample

In [ ]:
inference_result = test_workflow.run_sample_inference(
    model_run=model_run_resolution,
    checkpoint=checkpoint_selection,
    target=target_resolution,
    progress_session=test_model_progress,
)

## 8.2 Review model inference result

In [ ]:
display_review_sections(test_workflow.review_inference_result(inference_result))
if not inference_result.succeeded:
    raise RuntimeError(
        "Single-sample inference failed; stopping before comparison, visualization, reports, and publish."
    )

# 9. Runtime Console: Reference Comparison


## 9.1 Compare generated pose with reference keypoints


In [ ]:
if not inference_result.succeeded:
    raise RuntimeError("Inference failed; reference comparison is not meaningful.")

comparison = test_workflow.compare_reference_and_generated(
    target=target_resolution,
    inference=inference_result,
    progress_session=test_model_progress,
)

display_review_sections(
    test_workflow.review_reference_comparison_result(comparison)
)
if not comparison.succeeded:
    raise RuntimeError("Reference comparison failed; stopping before visualization, reports, and publish.")


# 9. Runtime Console: Visualization

## 9.1 Render generated pose and side-by-side video

In [ ]:
if not inference_result.succeeded:
    raise RuntimeError("Inference failed; visualization cannot be rendered.")
if not comparison.succeeded:
    raise RuntimeError("Reference comparison failed; visualization cannot be rendered.")

visualization_result = test_workflow.render_visualization(
    evidence=sample_evidence,
    inference=inference_result,
    progress_session=test_model_progress,
)

## 9.2 Review visualization result

In [ ]:
display_review_sections(test_workflow.review_visualization_result(visualization_result))

# 10. Runtime Console: Reports

## 10.1 Write test-model reports

In [ ]:
if not inference_result.succeeded or not comparison.succeeded or not visualization_result.succeeded:
    raise RuntimeError("Normal reports require successful inference, reference comparison, and visualization.")

report_result = test_workflow.write_reports(
    model_run=model_run_resolution,
    checkpoint=checkpoint_selection,
    target=target_resolution,
    evidence=sample_evidence,
    inference=inference_result,
    comparison=comparison,
    visualization=visualization_result,
    progress_session=test_model_progress,
)

## 10.2 Review report outputs

In [ ]:
display_review_sections(test_workflow.review_report_outputs(report_result))

# 11. Runtime Console: Publish and Verify

## 11.1 Build publish plan

In [ ]:
if not inference_result.succeeded or not comparison.succeeded or not visualization_result.succeeded:
    raise RuntimeError("Normal publish requires successful inference, reference comparison, and visualization.")

publish_plan = test_workflow.build_publish_plan(
    report_result=report_result,
    inference_result=inference_result,
    visualization_result=visualization_result,
)

## 11.2 Review publish plan

In [ ]:
display_review_sections(test_workflow.review_publish_plan(publish_plan))

## 11.3 Execute publish

In [ ]:
publish_execution = test_workflow.execute_publish(
    publish_plan,
    progress_session=test_model_progress,
)

## 11.4 Review publish execution

In [ ]:
display_review_sections(test_workflow.review_publish_execution(publish_execution))

## 11.5 Verify publish

In [ ]:
publish_verification = test_workflow.verify_publish(publish_execution)

## 11.6 Review publish verification

In [ ]:
display_review_sections(test_workflow.review_publish_verification(publish_verification))

## 11.7 Build publish result

In [ ]:
publish_result = test_workflow.build_publish_result(
    publish_plan,
    publish_execution,
    publish_verification,
)
if not publish_result.verification.succeeded:
    raise RuntimeError("Publish verification failed.")

## 11.8 Review publish result

In [ ]:
display_review_sections(test_workflow.review_publish_result(publish_result))

# 12. Final Summary

In [ ]:
final_result = test_workflow.build_final_result(
    model_run=model_run_resolution,
    checkpoint=checkpoint_selection,
    target=target_resolution,
    evidence=sample_evidence,
    inference=inference_result,
    comparison=comparison,
    visualization=visualization_result,
    reports=report_result,
    publish=publish_result,
)
display_review_sections(test_workflow.review_final_operator_summary(final_result))
print(f"MODEL_RUN_NAME: {model_run_resolution.model_run_name}")
print(f"Resolved model key: {model_run_resolution.model_key}")
print(f"CHECKPOINT_POLICY: {checkpoint_selection.policy.value}")
print(f"Checkpoint path: {checkpoint_selection.checkpoint_path}")
print(f"TARGET_SENTENCE_NAME: {test_request.target_sentence_name}")
print(f"Resolved test split sample: {target_resolution.resolved_sample_id}")
print(f"Manifest family: {model_run_resolution.manifest_family.family_id if model_run_resolution.manifest_family else None}")
print(f"Generated pose artifact: {inference_result.provider_result.generated_payload_path if inference_result.provider_result else None}")
print(f"Reference comparison status: {comparison.status}")
print(f"Reference-vs-generated video: {next((artifact.path for artifact in visualization_result.artifacts if artifact.label == 'reference_vs_generated_pose'), None)}")
print(f"Source-vs-generated side-by-side video: {next((artifact.path for artifact in visualization_result.artifacts if artifact.label == 'source_vs_generated_pose'), None)}")
print(f"Report paths: {[path.as_posix() for path in report_result.files]}")
print(f"Publish verification status: {publish_verification.succeeded}")
print("This is one-sample model test, not aggregate evaluation.")